# 04_compare_and_plots
Loads cached fold artifacts and builds model comparison tables and plots without retraining.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
metrics_root = ROOT / "artifacts" / "metrics"

rows = []
for model_dir in metrics_root.glob("*"):
    if not model_dir.is_dir():
        continue
    for fold_dir in model_dir.glob("fold_*"):
        metrics_path = fold_dir / "metrics.json"
        if not metrics_path.exists():
            continue
        with open(metrics_path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        rows.append(
            {
                "model": payload["model"],
                "fold": payload["fold"],
                "roc_auc": payload["screen_mode"]["roc_auc"],
                "pr_auc": payload["screen_mode"]["pr_auc"],
                "f1": payload["screen_mode"]["f1"],
                "recall": payload["screen_mode"]["recall"],
                "specificity": payload["screen_mode"]["specificity"],
                "fpr": payload["screen_mode"]["fpr"],
            }
        )

metrics_df = pd.DataFrame(rows)
summary = metrics_df.groupby("model").agg(["mean", "std"])
display(summary)

plot_df = metrics_df.groupby("model", as_index=False)["roc_auc"].mean().sort_values("roc_auc", ascending=False)
plt.figure(figsize=(10, 4))
plt.bar(plot_df["model"], plot_df["roc_auc"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Mean ROC-AUC")
plt.title("Model Comparison (Screening Mode)")
plt.tight_layout()
plt.show()